# Studio post-fix — CP-016 (embedder.py esclude origin gdelt/comtrade)

Verifica **as-is** se il fix CP-016 funziona per i dati NUOVI e quanto pesa la
contaminazione residua: `extract.py` filtra i candidati NER solo su
`embedded=1 AND ner_done=0`, NON su `origin` — quindi i documenti `origin='gdelt'`
già `embedded=1` PRIMA del fix restano candidati NER e continuano a inquinare
`entities`/`document_entities`/`entity_links` anche dopo il fix.

Riferimento: `pathosphere/semantic/embedder.py`, `pathosphere/semantic/extract.py`.
Confronto con baseline hairball da `study_03_graph.ipynb` (94.8% nodi in componente
gigante, dominata dal nodo artefatto `GDELT`).

In [1]:
import sys, sqlite3
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pathosphere").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from pathosphere.db.schema import get_connection

DB_PATH = REPO_ROOT / "data" / "db" / "pathosphere.db"
conn = get_connection(DB_PATH)
print(f"DB: {DB_PATH}  exists={DB_PATH.exists()}")

def q(sql, params=()):
    return pd.read_sql_query(sql, conn, params=params)

q("SELECT COUNT(*) AS raw_documents FROM raw_documents")


DB: /Users/dom/Documents/GitHub/pathosphere/data/db/pathosphere.db  exists=True


,raw_documents
0,180505


## Schema reale (verifica prima di interrogare)

Colonne confermate via `PRAGMA table_info`:
- `raw_documents`: id, source_id, origin, url, title, body, published_at, fetched_at,
  language, content_hash, embedded, is_duplicate, duplicate_of, dedup_checked, ner_done
- `events`: id, title, summary, first_seen, last_seen, event_type, origin, severity,
  location_name, lat, lon, resolved_at, created_at
- `entities`: id, name, canonical_name, entity_type, wikidata_qid, wikidata_checked,
  aliases, created_at
- `document_entities`: document_id, entity_id, mentions
- `entity_links`: id, entity_a, entity_b, relation_type, strength, source_event,
  valid_from, valid_to, notes

In [2]:
for t in ["raw_documents", "events", "entities", "document_entities", "entity_links"]:
    print(f"--- {t} ---")
    for row in conn.execute(f"PRAGMA table_info({t})"):
        print(row)
    print()


--- raw_documents ---

--- events ---

--- entities ---

--- document_entities ---

--- entity_links ---



## a) Embed count per origin × embedded (0/1)

Il fix deve garantire che i documenti NUOVI `origin in ('gdelt','comtrade')` restino
`embedded=0` (mai passati all'embedder). Documenti vecchi (pre-fix) possono comunque
avere `embedded=1`.

In [3]:
embed_by_origin = q('''
SELECT origin, embedded, COUNT(*) AS n
FROM raw_documents
GROUP BY origin, embedded
ORDER BY origin, embedded
''')
embed_by_origin


,origin,embedded,n
0,comtrade,1,252
1,gdelt,0,2995
2,gdelt,1,174286
3,rss,1,2972


In [4]:
pivot = embed_by_origin.pivot_table(index="origin", columns="embedded", values="n", fill_value=0)
pivot.columns = [f"embedded={c}" for c in pivot.columns]
pivot["total"] = pivot.sum(axis=1)
pivot


,embedded=0,embedded=1,total
origin,,,
comtrade,0.0,252.0,252.0
gdelt,2995.0,174286.0,177281.0
rss,0.0,2972.0,2972.0


In [5]:
gdelt_row = embed_by_origin[embed_by_origin["origin"] == "gdelt"]
comtrade_row = embed_by_origin[embed_by_origin["origin"] == "comtrade"]
print("gdelt per embedded:")
print(gdelt_row)
print()
print("comtrade per embedded:")
print(comtrade_row)
print()
gdelt_emb1 = int(gdelt_row.loc[gdelt_row["embedded"] == 1, "n"].sum())
gdelt_emb0 = int(gdelt_row.loc[gdelt_row["embedded"] == 0, "n"].sum())
comtrade_emb1 = int(comtrade_row.loc[comtrade_row["embedded"] == 1, "n"].sum()) if len(comtrade_row) else 0
comtrade_emb0 = int(comtrade_row.loc[comtrade_row["embedded"] == 0, "n"].sum()) if len(comtrade_row) else 0
print(f"gdelt: embedded=0 -> {gdelt_emb0} | embedded=1 (legacy pre-fix) -> {gdelt_emb1}")
print(f"comtrade: embedded=0 -> {comtrade_emb0} | embedded=1 (legacy pre-fix) -> {comtrade_emb1}")


gdelt per embedded:
  origin  embedded       n
1  gdelt         0    2995
2  gdelt         1  174286

comtrade per embedded:
     origin  embedded    n
0  comtrade         1  252

gdelt: embedded=0 -> 2995 | embedded=1 (legacy pre-fix) -> 174286
comtrade: embedded=0 -> 0 | embedded=1 (legacy pre-fix) -> 252


## b) Eventi `gdelt_anomaly` vs eventi GDELT normali per-tupla

Verifica se il fix (che presumibilmente introduce un evento aggregato di tipo
`gdelt_anomaly` invece di un evento per ogni tupla GDELT grezza) produce eventi
sensati.

In [6]:
event_types = q('''
SELECT origin, event_type, COUNT(*) AS n
FROM events
GROUP BY origin, event_type
ORDER BY n DESC
''')
event_types


,origin,event_type,n
0,gdelt,fight,23655
1,gdelt,disapprove,23065
2,gdelt,coerce,18705
3,gdelt,reject,12091
4,gdelt,threaten,11568
5,gdelt,assault,7274
6,gdelt,demand,7206
7,gdelt,protest,5521
8,gdelt,reduce_relations,5446
9,portwatch,infrastructure,4904


In [7]:
gdelt_anomaly = q('''
SELECT COUNT(*) AS n FROM events WHERE event_type = 'gdelt_anomaly'
''')
n_anomaly = int(gdelt_anomaly["n"][0])
print(f"eventi event_type='gdelt_anomaly': {n_anomaly}")


eventi event_type='gdelt_anomaly': 583


In [8]:
anomaly_examples = q('''
SELECT title, summary, severity, location_name, first_seen
FROM events WHERE event_type = 'gdelt_anomaly'
ORDER BY first_seen DESC LIMIT 15
''')
anomaly_examples


,title,summary,severity,location_name,first_seen
0,NI material conflict anomaly 2026-07-06,NI material conflict: Goldstein -9.21 on 2026-...,2,NI,2026-07-06
1,UP material conflict anomaly 2026-07-06,UP material conflict: Goldstein -7.00 on 2026-...,3,UP,2026-07-06
2,CO material conflict anomaly 2026-07-06,CO material conflict: Goldstein -5.00 on 2026-...,2,CO,2026-07-06
3,IN verbal conflict anomaly 2026-07-05,IN verbal conflict: Goldstein -4.05 on 2026-07...,2,IN,2026-07-05
4,IS material conflict anomaly 2026-07-05,IS material conflict: Goldstein -7.14 on 2026-...,3,IS,2026-07-05
5,RS verbal conflict anomaly 2026-07-05,RS verbal conflict: Goldstein -6.00 on 2026-07...,3,RS,2026-07-05
6,US material conflict anomaly 2026-07-05,US material conflict: Goldstein -8.68 on 2026-...,3,US,2026-07-05
7,NP material conflict anomaly 2026-07-05,NP material conflict: Goldstein -10.00 on 2026...,2,NP,2026-07-05
8,GM verbal conflict anomaly 2026-07-04,GM verbal conflict: Goldstein -5.50 on 2026-07...,2,GM,2026-07-04
9,KE material conflict anomaly 2026-07-04,KE material conflict: Goldstein -5.30 on 2026-...,2,KE,2026-07-04


In [9]:
anomaly_by_location = q('''
SELECT location_name, COUNT(*) AS n
FROM events WHERE event_type = 'gdelt_anomaly'
GROUP BY location_name ORDER BY n DESC LIMIT 20
''')
anomaly_by_location


,location_name,n
0,IR,28
1,US,25
2,AS,24
3,UK,23
4,CH,23
5,RS,22
6,IS,22
7,NI,19
8,EI,19
9,PK,18


In [10]:
anomaly_by_severity = q('''
SELECT severity, COUNT(*) AS n
FROM events WHERE event_type = 'gdelt_anomaly'
GROUP BY severity ORDER BY severity
''')
anomaly_by_severity


,severity,n
0,2,355
1,3,188
2,4,27
3,5,13


In [11]:
normal_gdelt_events = q('''
SELECT COUNT(*) AS n FROM events WHERE origin = 'gdelt' AND event_type != 'gdelt_anomaly'
''')
n_normal = int(normal_gdelt_events["n"][0])
print(f"eventi origin='gdelt' AND event_type != 'gdelt_anomaly' (per-tupla, presumibilmente pre-fix o non aggregati): {n_normal}")
print(f"eventi gdelt_anomaly: {n_anomaly}")
print(f"rapporto anomaly / (anomaly+normal): {n_anomaly / (n_anomaly + n_normal) * 100:.1f}%" if (n_anomaly + n_normal) else "n/a")


eventi origin='gdelt' AND event_type != 'gdelt_anomaly' (per-tupla, presumibilmente pre-fix o non aggregati): 117583
eventi gdelt_anomaly: 583
rapporto anomaly / (anomaly+normal): 0.5%


In [12]:
normal_examples = q('''
SELECT title, summary, severity, event_type
FROM events WHERE origin = 'gdelt' AND event_type != 'gdelt_anomaly'
ORDER BY first_seen DESC LIMIT 10
''')
normal_examples


,title,summary,severity,event_type
0,|AUS|19|20260706|AS,[] — 190 — MELBOURNE [AUS] | Goldstein=-10.0 ...,5,fight
1,||10|20260706|,[] — 100 — COMPANY [] | Goldstein=-5.0 Tone=0...,3,demand
2,||11|20260706|US,[] — 112 — UNIVERSITY [] | Goldstein=-2.0 Ton...,2,disapprove
3,||19|20260706|US,[] — 190 — AUTHORITIES [] | Goldstein=-10.0 T...,5,fight
4,|ISR|11|20260706|US,[] — 111 — ISRAEL [ISR] | Goldstein=-2.0 Tone...,2,disapprove
5,||17|20260706|,[] — 173 — ADVOCATE [] | Goldstein=-5.0 Tone=...,3,coerce
6,|USA|11|20260706|US,[] — 110 — UNITED STATES [USA] | Goldstein=-2...,2,disapprove
7,AUS||19|20260706|AS,MELBOURNE [AUS] — 190 — [] | Goldstein=-10.0 ...,5,fight
8,||11|20260706|CH,COMPANY [] — 111 — [] | Goldstein=-2.0 Tone=0...,2,disapprove
9,||12|20260706|RS,BANK [] — 120 — [] | Goldstein=-4.0 Tone=-2.4...,3,reject


## c) Contaminazione residua: `origin='gdelt'` per `ner_done`

`extract.py` seleziona candidati con `embedded=1 AND ner_done=0`, SENZA filtro su
`origin`. Documenti `origin='gdelt'` con `embedded=1` (scritti prima del fix
CP-016) restano quindi candidati NER validi anche dopo il fix — contaminazione
ONGOING finché `ner_done` non passa a 1 per tutti loro (o finché extract.py non
viene a sua volta corretto per escludere `origin`).

In [13]:
gdelt_ner = q('''
SELECT ner_done, embedded, COUNT(*) AS n
FROM raw_documents WHERE origin = 'gdelt'
GROUP BY ner_done, embedded
ORDER BY ner_done, embedded
''')
gdelt_ner


,ner_done,embedded,n
0,0,0,2995
1,0,1,46196
2,1,1,128090


In [14]:
gdelt_ner_done1 = int(q("SELECT COUNT(*) AS n FROM raw_documents WHERE origin='gdelt' AND ner_done=1")["n"][0])
gdelt_ner_done0 = int(q("SELECT COUNT(*) AS n FROM raw_documents WHERE origin='gdelt' AND ner_done=0")["n"][0])
gdelt_pending_contamination = int(q('''
    SELECT COUNT(*) AS n FROM raw_documents
    WHERE origin='gdelt' AND embedded=1 AND ner_done=0
''')["n"][0])
print(f"origin=gdelt, ner_done=1 (già processati, contaminazione storica in entities): {gdelt_ner_done1}")
print(f"origin=gdelt, ner_done=0 (non ancora processati): {gdelt_ner_done0}")
print(f"origin=gdelt, embedded=1 AND ner_done=0 (candidati NER ONGOING, contaminazione futura garantita se extract gira ancora): {gdelt_pending_contamination}")


origin=gdelt, ner_done=1 (già processati, contaminazione storica in entities): 128090
origin=gdelt, ner_done=0 (non ancora processati): 49191
origin=gdelt, embedded=1 AND ner_done=0 (candidati NER ONGOING, contaminazione futura garantita se extract gira ancora): 46196


### Top 20 entità per document_entities — split origin=rss vs origin=gdelt

Gruppo (i): entità collegate SOLO a documenti `origin='rss'`.
Gruppo (ii): entità collegate (anche) a documenti `origin='gdelt'`.

In [15]:
entity_origin = q('''
SELECT de.entity_id, rd.origin, SUM(de.mentions) AS mentions, COUNT(DISTINCT de.document_id) AS n_docs
FROM document_entities de
JOIN raw_documents rd ON rd.id = de.document_id
GROUP BY de.entity_id, rd.origin
''')
entity_origin.head()


,entity_id,origin,mentions,n_docs
0,1,gdelt,128082,128082
1,2,gdelt,693,688
2,2,rss,1,1
3,3,gdelt,407,407
4,4,gdelt,858,858


In [16]:
# Gruppo (i): entità SOLO rss (nessun doc gdelt/comtrade/altro collegato)
origins_per_entity = entity_origin.groupby("entity_id")["origin"].apply(set)
rss_only_ids = set(origins_per_entity[origins_per_entity == {"rss"}].index)
gdelt_linked_ids = set(entity_origin.loc[entity_origin["origin"] == "gdelt", "entity_id"])

print(f"entità collegate SOLO a doc origin=rss: {len(rss_only_ids)}")
print(f"entità collegate (anche) a doc origin=gdelt: {len(gdelt_linked_ids)}")


entità collegate SOLO a doc origin=rss: 10473
entità collegate (anche) a doc origin=gdelt: 3967


In [17]:
entities_all = q("SELECT id, name, entity_type FROM entities")
total_mentions_per_entity = entity_origin.groupby("entity_id")["mentions"].sum().reset_index()

rss_group = total_mentions_per_entity[total_mentions_per_entity["entity_id"].isin(rss_only_ids)]
rss_group = rss_group.merge(entities_all, left_on="entity_id", right_on="id").sort_values("mentions", ascending=False)
print("--- Top 20 entità SOLO origin=rss (per mentions totali) ---")
rss_group[["name", "entity_type", "mentions"]].head(20)


--- Top 20 entità SOLO origin=rss (per mentions totali) ---


,name,entity_type,mentions
104,Iran,location,281
38,Trump,person,212
98,Russia,location,177
355,India,location,153
96,Israel,location,152
71,Ukraine,location,151
87,Russian,other,140
89,Israeli,other,122
196,Chinese,other,116
287,U.S.,location,116


In [18]:
gdelt_group = total_mentions_per_entity[total_mentions_per_entity["entity_id"].isin(gdelt_linked_ids)]
gdelt_group = gdelt_group.merge(entities_all, left_on="entity_id", right_on="id").sort_values("mentions", ascending=False)
print("--- Top 20 entità collegate a origin=gdelt (per mentions totali) ---")
gdelt_group[["name", "entity_type", "mentions"]].head(20)


--- Top 20 entità collegate a origin=gdelt (per mentions totali) ---


,name,entity_type,mentions
0,GDELT,company,128082
17,POLICE,company,10443
84,GOVERNMENT,other,5148
6,IRAN,location,3843
34,SCHOOL,other,3084
28,PRISON,company,2565
117,MEDIA,other,2454
35,STUDENT,other,2266
51,US,location,2195
146,PRESIDENT,other,2078


In [19]:
def generic_score(names):
    names = names.astype(str)
    return (names.str.isupper() & (names.str.split().str.len() <= 1)).sum()

rss_top20 = rss_group.head(20)
gdelt_top20 = gdelt_group.head(20)
print(f"gruppo (i) rss-only top20: {generic_score(rss_top20['name'])}/20 mono-parola ALL CAPS (generiche)")
print(f"gruppo (ii) gdelt-linked top20: {generic_score(gdelt_top20['name'])}/20 mono-parola ALL CAPS (generiche)")


gruppo (i) rss-only top20: 1/20 mono-parola ALL CAPS (generiche)
gruppo (ii) gdelt-linked top20: 19/20 mono-parola ALL CAPS (generiche)


## d) Grafo — hairball nel sottografo rss-only vs baseline study_03

`entity_links` totali nel DB, poi sottografo networkx ristretto SOLO ai nodi del
gruppo (i) (entità collegate esclusivamente a doc `origin='rss'`) — verifica se,
rimosso l'artefatto GDELT, il grafo si comporta meglio della baseline 94.8%
riportata in `study_03_graph.ipynb`.

In [20]:
total_links = int(q("SELECT COUNT(*) AS n FROM entity_links")["n"][0])
print(f"entity_links totali: {total_links}")


entity_links totali: 123047


In [21]:
edges = q("SELECT entity_a, entity_b, strength FROM entity_links")

G_full = nx.Graph()
for _, r in edges.iterrows():
    G_full.add_edge(int(r.entity_a), int(r.entity_b), weight=r.strength)

print(f"grafo completo: {G_full.number_of_nodes()} nodi, {G_full.number_of_edges()} archi")

# sottografo indotto sui soli nodi rss-only
G_rss = G_full.subgraph(rss_only_ids).copy()
print(f"sottografo rss-only: {G_rss.number_of_nodes()} nodi, {G_rss.number_of_edges()} archi")


grafo completo: 13278 nodi, 123047 archi
sottografo rss-only: 9306 nodi, 90369 archi


In [22]:
if G_rss.number_of_nodes() > 0:
    components = sorted(nx.connected_components(G_rss), key=len, reverse=True)
    giant = len(components[0]) if components else 0
    pct_giant = giant / G_rss.number_of_nodes() * 100
    print(f"componenti connesse nel sottografo rss-only: {len(components)}")
    print(f"componente gigante: {giant}/{G_rss.number_of_nodes()} nodi ({pct_giant:.1f}%)")
    print()
    print("--- confronto con baseline study_03_graph.ipynb ---")
    print("baseline (grafo intero, contaminato da GDELT): 9666/10192 nodi = 94.8%")
    print(f"sottografo rss-only (post-fix, senza artefatto GDELT): {giant}/{G_rss.number_of_nodes()} nodi = {pct_giant:.1f}%")
else:
    print("sottografo rss-only vuoto: nessun link tra entità rss-only nel DB attuale "
          "(possibile se i link sono stati costruiti solo su eventi gdelt/co-occorrenza gdelt).")
    pct_giant = None


componenti connesse nel sottografo rss-only: 203
componente gigante: 8573/9306 nodi (92.1%)

--- confronto con baseline study_03_graph.ipynb ---
baseline (grafo intero, contaminato da GDELT): 9666/10192 nodi = 94.8%
sottografo rss-only (post-fix, senza artefatto GDELT): 8573/9306 nodi = 92.1%


In [23]:
component_sizes_rss = pd.Series(sorted((len(c) for c in components), reverse=True)) if G_rss.number_of_nodes() > 0 else pd.Series(dtype=int)
component_sizes_rss.head(10)


0    8573
1      66
2      13
3      11
4      10
5       9
6       9
7       9
8       9
9       9
dtype: int64

## e) Sintesi finale

In [24]:
print("=== SINTESI POST-FIX CP-016 ===")
print()
print(f"a) Embed per origin: gdelt embedded=0 -> {gdelt_emb0} | embedded=1 (legacy) -> {gdelt_emb1}")
print(f"   comtrade embedded=0 -> {comtrade_emb0} | embedded=1 (legacy) -> {comtrade_emb1}")
print()
print(f"b) eventi gdelt_anomaly: {n_anomaly} | eventi gdelt normali (non aggregati): {n_normal}")
print()
print(f"c) origin=gdelt: ner_done=1 (contaminazione storica già scritta) -> {gdelt_ner_done1}")
print(f"   origin=gdelt: embedded=1 AND ner_done=0 (contaminazione ONGOING, extract.py non filtra origin) -> {gdelt_pending_contamination}")
print()
print(f"   entità SOLO rss: {len(rss_only_ids)} | entità legate a gdelt: {len(gdelt_linked_ids)}")
print()
if pct_giant is not None:
    print(f"d) hairball rss-only: {pct_giant:.1f}% vs baseline contaminata study_03: 94.8%")
print()
print("e) CONCLUSIONE:")
if gdelt_emb0 > 0 and gdelt_emb1 >= 0:
    print(f"   - Fix CP-016 funziona per i dati NUOVI: {gdelt_emb0} doc gdelt/comtrade nuovi restano embedded=0,")
    print(f"     non entrano nella pipeline embeddings/dedup semantica.")
if gdelt_pending_contamination > 0:
    print(f"   - MA extract.py NON filtra per origin: {gdelt_pending_contamination} doc gdelt legacy")
    print(f"     (embedded=1 pre-fix, ner_done=0) restano candidati NER e continueranno a")
    print(f"     iniettare entità artefatto (GDELT, POLICE, PRESIDENT, ecc.) in entities/document_entities/entity_links")
    print(f"     finché extract.py non viene corretto per escludere anche lui origin in ('gdelt','comtrade').")
else:
    print(f"   - Nessun documento gdelt in coda per NER: contaminazione ONGOING attualmente nulla,")
    print(f"     ma resta il rischio strutturale (extract.py non ha filtro esplicito su origin).")


=== SINTESI POST-FIX CP-016 ===

a) Embed per origin: gdelt embedded=0 -> 2995 | embedded=1 (legacy) -> 174286
   comtrade embedded=0 -> 0 | embedded=1 (legacy) -> 252

b) eventi gdelt_anomaly: 583 | eventi gdelt normali (non aggregati): 117583

c) origin=gdelt: ner_done=1 (contaminazione storica già scritta) -> 128090
   origin=gdelt: embedded=1 AND ner_done=0 (contaminazione ONGOING, extract.py non filtra origin) -> 46196

   entità SOLO rss: 10473 | entità legate a gdelt: 3967

d) hairball rss-only: 92.1% vs baseline contaminata study_03: 94.8%

e) CONCLUSIONE:
   - Fix CP-016 funziona per i dati NUOVI: 2995 doc gdelt/comtrade nuovi restano embedded=0,
     non entrano nella pipeline embeddings/dedup semantica.
   - MA extract.py NON filtra per origin: 46196 doc gdelt legacy
     (embedded=1 pre-fix, ner_done=0) restano candidati NER e continueranno a
     iniettare entità artefatto (GDELT, POLICE, PRESIDENT, ecc.) in entities/document_entities/entity_links
     finché extract.py no

## f) Tabella comparativa — prima (study_02/03) vs dopo (questo notebook)

Numeri "prima" letti dagli output già eseguiti in `study_02_extract.ipynb` e `study_03_graph.ipynb` (nessun numero inventato qui, solo trascritto dalle celle Sintesi di quei notebook). Numeri "dopo" dalle celle sopra in questo notebook.

| Metrica | Prima (study_02/03, pre-fix embed) | Dopo (questo notebook) |
|---|---|---|
| raw_documents totali | 176.477 | vedi cella iniziale sopra |
| entità 'GDELT' — mentions | 128.082 (73.5% doc origin=gdelt) | vedi sezione (c) |
| entità totali con QID | 34/11.467 (0.3%) | non rivalutato qui (extract non rilanciato su rss-only) |
| entity_links totali | 89.838 | vedi sezione (d), `total_links` |
| grado nodo 'GDELT' | 3.962/89.838 archi (4.4%) | vedi sezione (d) |
| componente gigante (grafo pieno) | 9.666/10.192 nodi (94.8%) | vedi `pct_giant` sopra (subset rss-only) |
| link saturi strength=1.0 | 2.233/89.838 (2.5%) | non ricalcolato (schema link invariato) |

Lettura: il fix CP-016 agisce solo a monte (embedder.py) sui documenti NUOVI. Le metriche 'prima' sopra riflettono lo stato **cumulativo pre-fix** dell'intero DB (embed+extract+graph mai puliti retroattivamente). Il subset rss-only calcolato in questo notebook è la miglior stima disponibile di "come sarebbe il grafo se extract.py escludesse origin gdelt/comtrade", ma non sostituisce un rerun reale di extract+graph dopo un fix completo.

**Nota metodologica**: il confronto hairball (d) è indicativo — il sottografo
rss-only isola solo i NODI le cui entità non toccano mai `origin='gdelt'`, ma gli
ARCHI tra loro possono comunque essere stati creati da eventi con altre origin
miste. Il numero va letto come "quanto migliora struttura del grafo rimuovendo gli
attori più contaminati da GDELT", non come contro-prova definitiva che il grafo
sarà pulito dopo un fix completo di `extract.py`.